# RevCoCo Revenue Engine V3 — ML Model Comparison

**Objective:** Compare XGBoost, Logistic Regression, and kNN for at-risk receivables prediction.

**Metrics:** Accuracy, Precision, Recall, Confusion Matrix

**Data:** `KIPI_REVCOCO.MVP.receivables_history` (sampled 100K rows)

**Features:** invoice_amount, past_dso_avg + engineered features

In [ ]:
%%sql -r dataframe_1
SELECT
    invoice_amount,
    past_dso_avg,
    is_late
FROM KIPI_REVCOCO.MVP.receivables_history
SAMPLE (100000 ROWS)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    confusion_matrix
)
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = sql_data.to_pandas()

df['LOG_AMOUNT'] = np.log1p(df['INVOICE_AMOUNT'])
df['AMOUNT_PER_DSO_DAY'] = df['INVOICE_AMOUNT'] / df['PAST_DSO_AVG'].replace(0, np.nan)
df['AMOUNT_PER_DSO_DAY'] = df['AMOUNT_PER_DSO_DAY'].fillna(0)
df['DSO_SQUARED_SCALED'] = (df['PAST_DSO_AVG'] ** 2) / 1000.0
df['AMOUNT_NORMALIZED'] = (df['INVOICE_AMOUNT'] - 15000) / 15000.0
df['HIGH_RISK_COMBO'] = ((df['INVOICE_AMOUNT'] > 25000) & (df['PAST_DSO_AVG'] > 35)).astype(int)

features = [
    'INVOICE_AMOUNT', 'PAST_DSO_AVG', 'LOG_AMOUNT',
    'AMOUNT_PER_DSO_DAY', 'DSO_SQUARED_SCALED',
    'AMOUNT_NORMALIZED', 'HIGH_RISK_COMBO'
]

X = df[features]
y = df['IS_LATE'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape[0]:,} rows")
print(f"Test set:     {X_test.shape[0]:,} rows")
print(f"Features:     {len(features)}")
print(f"Late rate:    {y.mean():.2%}")

## Model Training
Train all three models: XGBoost (unscaled), Logistic Regression (scaled), kNN (scaled)

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
print("XGBoost trained.")

In [ ]:
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)
print("Logistic Regression trained.")

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=7)
knn_model.fit(X_train_scaled, y_train)
knn_pred = knn_model.predict(X_test_scaled)
print("kNN (k=7) trained.")

## Model Comparison

In [ ]:
models = {
    'XGBoost': xgb_pred,
    'Logistic Regression': lr_pred,
    'kNN (k=7)': knn_pred
}

results = {}
for name, pred in models.items():
    results[name] = {
        'Accuracy': round(accuracy_score(y_test, pred), 4),
        'Precision': round(precision_score(y_test, pred), 4),
        'Recall': round(recall_score(y_test, pred), 4)
    }

comparison_df = pd.DataFrame(results).T
comparison_df.index.name = 'Model'
comparison_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, pred) in enumerate(models.items()):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
        xticklabels=['Not Late', 'Late'],
        yticklabels=['Not Late', 'Late']
    )
    axes[idx].set_title(f"{name}\nAccuracy: {results[name]['Accuracy']:.4f}")
    axes[idx].set_ylabel('Actual')
    axes[idx].set_xlabel('Predicted')

plt.suptitle(
    'Confusion Matrix Comparison — At-Risk Receivables Prediction',
    fontsize=14, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.show()

In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall']
x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))

for i, (name, vals) in enumerate(results.items()):
    values = [vals[m] for m in metrics]
    bars = ax.bar(x + i * width, values, width, label=name)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.4f}', ha='center', va='bottom', fontsize=9)

ax.set_ylabel('Score')
ax.set_title('Model Comparison — Accuracy, Precision, Recall')
ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_ylim(0, 1.1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
importance = pd.Series(
    xgb_model.feature_importances_,
    index=features
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
importance.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('XGBoost Feature Importance')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

## Summary

| Algorithm | Strengths | Notes |
|---|---|---|
| **XGBoost** | Best accuracy, handles non-linear patterns | Gradient boosting, tree-based |
| **Logistic Regression** | Fast, interpretable, good baseline | Linear decision boundary |
| **kNN (k=7)** | Non-parametric, no training phase | Sensitive to feature scaling and k value |